<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/01c_remove_near_duplicates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Remove the leaking near-duplicates before retraining

Notebook 01b found 157 near-duplicate pairs across the splits. This removes the **leaking copy** of each pair:
the training copy always, and the validation copy when its partner is in test. **The 2,992 test images stay exactly the same**,
so results before and after retraining are measured on the same test set. Every flagged pair is handled, real duplicates and false matches
alike; a few hundred training images out of ~14,000 is a negligible loss.

Steps: (1) archive every current result on Drive, (2) remove the leaking copies from the split lists, (3) confirm no cross-split pairs remain,
(4) save the cleaned data as the new `harmonized.zip`. Then rerun notebooks 02 → 03 → 04 → 05 as usual: they read the new zip automatically.

**No GPU needed**: Runtime > Change runtime type > **CPU**. About 5-10 minutes, mostly copying to Drive.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e .

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
ZIP, PAIRS = f"{D}/harmonized.zip", f"{D}/data_quality/near_duplicates.csv"
for p in (ZIP, PAIRS):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found: run notebook 01 (step 8) and notebook 01b first")
print("found harmonized.zip and the near-duplicate report")

## 3. Archive the current results
Copies the current data zip and every result folder to `thesis-data/archive_split_v1/` **before anything is changed**, so the earlier
models and numbers stay available for comparison. Runs only once: if the archive already exists it is left untouched.
Needs roughly 0.5-1 GB of Drive space (mostly the model checkpoints).

In [ ]:
import shutil, json, datetime
ARCHIVE = f"{D}/archive_split_v1"
FOLDERS = ["models", "stress_test", "watermark", "provenance", "evaluation", "data_quality"]
if os.path.exists(f"{ARCHIVE}/ARCHIVE.json"):
    print(f"already archived on {json.load(open(f'{ARCHIVE}/ARCHIVE.json'))['archived_at']}; leaving it as it is")
else:
    os.makedirs(ARCHIVE, exist_ok=True)
    shutil.copy2(ZIP, f"{ARCHIVE}/harmonized.zip")
    copied = ["harmonized.zip"]
    for name in FOLDERS:
        if os.path.isdir(f"{D}/{name}"):
            shutil.copytree(f"{D}/{name}", f"{ARCHIVE}/{name}", dirs_exist_ok=True)
            copied.append(name)
    json.dump({"archived_at": datetime.datetime.now().isoformat(timespec="seconds"), "contents": copied,
               "note": "results on the original splits, before removing near-duplicate leaking copies"},
              open(f"{ARCHIVE}/ARCHIVE.json", "w"), indent=1)
    print("archived:", ", ".join(copied))
!du -sh $ARCHIVE

## 4. Remove the leaking copies

In [ ]:
!unzip -qo $ZIP -d .
!mkdir -p $D/data_quality && rm -rf outputs/data_quality && ln -s $D/data_quality outputs/data_quality
!python scripts/remove_near_duplicates.py --pairs outputs/data_quality/near_duplicates.csv

## 5. Confirm no cross-split near-duplicates remain
Runs the same check as notebook 01b on the cleaned splits (report in `thesis-data/data_quality/after_removal/`). It should find **0 pairs**.

In [ ]:
!python scripts/check_near_duplicates.py --out outputs/data_quality/after_removal
after = json.load(open("outputs/data_quality/after_removal/near_duplicates_summary.json"))
assert after["n_pairs"] == 0, f"{after['n_pairs']} pairs remain: don't save the zip, check the output above"
assert after["split_sizes"]["test"] == 2992, "the test set changed: don't save the zip"
print("clean: no cross-split near-duplicates; test set unchanged at", after["split_sizes"]["test"], "images")

## 6. Save the cleaned data as the new `harmonized.zip`

In [ ]:
assert os.path.exists(f"{ARCHIVE}/harmonized.zip"), "archive missing: run section 3 first"
!rm -f /content/harmonized_clean.zip && zip -qr /content/harmonized_clean.zip data/harmonized data/processed
shutil.copy2("/content/harmonized_clean.zip", ZIP)
!ls -lh $ZIP $ARCHIVE/harmonized.zip
print("saved; notebooks 02-05 will now use the cleaned splits")

## 7. Next: rerun 02 → 03 → 04 → 05
Wait a few minutes for Drive to sync the new zip, then run each notebook top to bottom and save it to GitHub (File > Save a copy in GitHub):

1. **02** (GPU): retrains ResNet-18, ResNet-50 and EfficientNet-B0 and reruns their stress tests. The longest step.
2. **03** (CPU): recalibrates the watermark threshold on the cleaned validation set.
3. **04** (CPU): provenance stress test.
4. **05** (GPU): sections 1-7, the fusion evaluation and the no-watermark ablation. Section 6 can be skipped (it only recomputes section 4's metrics).

Notebook 05b doesn't need rerunning: the leaking copies are gone, so there's nothing left to exclude.